- agent_counter

- Özellik Atama

- Panda DataFrame Kullanımı


In [1]:
!pip install mesa[rec]
!pip install seaborn

# Has multi-dimensional arrays and matrices.
# Has a large collection of mathematical functions to operate on these arrays.
import numpy as np

# Data manipulation and analysis.
import pandas as pd

# Data visualization tools.
import seaborn as sns

import mesa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.8/265.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.6 MB/s eta 0:00:00


In [32]:

from mesa.space import MultiGrid
from mesa.datacollection import DataCollector
import random

# Ajan sınıfını tanımlayalım
class MyAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.active = random.choice([True, False])
        self.wealth = random.randint(0, 100)
        self.group = "A" if self.unique_id % 2 == 0 else "B"  # Çift-ID'liler "A" grubu


    def say_hi(self):
        print(
            f"Merhaba! Ben {self.unique_id}, "
            f"Servetim: {self.wealth}, "
            f"Aktif mi? {self.active}, "
            f"Grubum: {self.group}"
        )


    def step(self):
        # Ajanın her adımda yapacağı işlemler
        print(f"Ajan ID: {self.unique_id}- Servet: {self.wealth}")
        self.say_hi()

# Model sınıfını tanımlayalım
class MyModel(mesa.Model):
    def __init__(self, N, width, height):
        super().__init__()
        self.num_agents = N
        self.grid = MultiGrid(width, height, True)

        self.agent_counter = self.num_agents  # ID sayacı




        # Ajanları oluşturalım
        for i in range(self.num_agents):
            agent = MyAgent(self)
            self.agents.add(agent)

            # Ajanları rastgele bir hücreye yerleştirelim
            x = self.random.randrange(self.grid.width)
            y = self.random.randrange(self.grid.height)
            self.grid.place_agent(agent, (x, y))

        # Veri toplamak için DataCollector kullanabiliriz (opsiyonel)
        self.datacollector = DataCollector()





    # Modelde filtreleme:
    def get_group_a(self):
        return self.agents.select(lambda agent: agent.group == "A")


    # Dinamik Ajan Ekleyip Çıkarma

    def dynamic_agents(self):
        # diğer piyasalara ilişkin kötü haberler arttığında ve mevcut piyasaya ilşkin iyi haberler arttığında, piyasaya ajan girişi diğer zamanlardaki
        # girişlerden daha çok olacaktır. Bu nedenle piyasaya gelen haberlere göre ajan girişi olacak
        if random.random() > 0.5: #iyi haber geldi, 100 yeni ajan girdi
            # Yeni ajan ekle
            for i in range(int(0.25 * len(self.agents))):   # yeni giriş yapan ajanların sayısını, mevcut ajan sayısının belli bir oranı olarak ayarlanabilir.
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
                self.agent_counter += 1
            print(f"Yeni eklenen ajan SAYISI: {(int(0.25 * len(self.agents)))}")
        else: # iyi haber yok, 10 yeni ajan girdi
            # Yeni ajan ekle
            for i in range(int(0.10 * len(self.agents))):
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
                self.agent_counter += 1
            print(f"Yeni eklenen ajan sayısİ: {(int(0.10 * len(self.agents)))}")

        print(f"Güncel Ajan SAyısı: {len(self.agents)}")

        """
        # 5 ID'li ajanı sil
        agent_to_remove = next(a for a in self.agents if a.unique_id == 5)
        self.agents.remove(agent_to_remove)
        """

    def get_agents_by_wealth(self, min_wealth):
        return [a for a in self.agents if a.wealth >= min_wealth]

    def count_agents(self):
        return len(list(self.agents))


    def step(self):
        # Modelin her adımda yapacağı işlemler
        self.agents.do("step")
        self.dynamic_agents()

        self.datacollector.collect(self)

# Modeli oluşturalım (100 ajan, 10x10 grid)
model = MyModel(10, 10, 10)

# Tüm ajanları AgentSet olarak almak için:
all_agents = model.agents

# AgentSet'i kontrol edelim
print(f"Toplam ajan sayısı: {len(all_agents)}")
print(f"İlk ajanın ID'si: {all_agents[0].unique_id}")

# Modeli birkaç adım çalıştıralım
for i in range(10):
    print(f"------Adım {i+1}--------")
    model.step()

Toplam ajan sayısı: 10
İlk ajanın ID'si: 1
------Adım 1--------
Ajan ID: 1- Servet: 20
Merhaba! Ben 1, Servetim: 20, Aktif mi? False, Grubum: B
Ajan ID: 2- Servet: 15
Merhaba! Ben 2, Servetim: 15, Aktif mi? True, Grubum: A
Ajan ID: 3- Servet: 46
Merhaba! Ben 3, Servetim: 46, Aktif mi? False, Grubum: B
Ajan ID: 4- Servet: 20
Merhaba! Ben 4, Servetim: 20, Aktif mi? True, Grubum: A
Ajan ID: 5- Servet: 39
Merhaba! Ben 5, Servetim: 39, Aktif mi? True, Grubum: B
Ajan ID: 6- Servet: 40
Merhaba! Ben 6, Servetim: 40, Aktif mi? True, Grubum: A
Ajan ID: 7- Servet: 64
Merhaba! Ben 7, Servetim: 64, Aktif mi? False, Grubum: B
Ajan ID: 8- Servet: 99
Merhaba! Ben 8, Servetim: 99, Aktif mi? True, Grubum: A
Ajan ID: 9- Servet: 34
Merhaba! Ben 9, Servetim: 34, Aktif mi? True, Grubum: B
Ajan ID: 10- Servet: 78
Merhaba! Ben 10, Servetim: 78, Aktif mi? False, Grubum: A
Yeni eklenen ajan SAYISI: 3
Güncel Ajan SAyısı: 12
------Adım 2--------
Ajan ID: 1- Servet: 20
Merhaba! Ben 1, Servetim: 20, Aktif mi? False

In [4]:
# Grid Üzerinden Erişim (Eğer Ajanlar Grid'deyse)

# Örnek: (5, 5) koordinatındaki ajanları al
cell_agents = model.grid.get_cell_list_contents([(5, 5)])
for agent in cell_agents:
    print(f"Ajan {agent.unique_id} bu hücrede.")


In [7]:
# Ajanlara özellik atama
for agent in model.agents:
    agent.age = random.randint(18, 80)  # Her ajana rastgele yaş ekle
    print(f"Ajan {agent.unique_id}, Yaş: {agent.age}")

Ajan 1, Yaş: 72
Ajan 2, Yaş: 63
Ajan 3, Yaş: 54
Ajan 4, Yaş: 24
Ajan 5, Yaş: 45
Ajan 6, Yaş: 45
Ajan 7, Yaş: 55
Ajan 8, Yaş: 45
Ajan 9, Yaş: 51
Ajan 10, Yaş: 27
Ajan 11, Yaş: 76
Ajan 12, Yaş: 27
Ajan 13, Yaş: 31
Ajan 14, Yaş: 62
Ajan 15, Yaş: 34
Ajan 16, Yaş: 73
Ajan 17, Yaş: 35
Ajan 18, Yaş: 45
Ajan 19, Yaş: 46
Ajan 20, Yaş: 48
Ajan 21, Yaş: 20
Ajan 22, Yaş: 51
Ajan 23, Yaş: 34
Ajan 24, Yaş: 24
Ajan 25, Yaş: 60
Ajan 26, Yaş: 24
Ajan 27, Yaş: 35
Ajan 28, Yaş: 61
Ajan 29, Yaş: 24
Ajan 30, Yaş: 18
Ajan 31, Yaş: 35
Ajan 32, Yaş: 47
Ajan 33, Yaş: 39
Ajan 34, Yaş: 19
Ajan 35, Yaş: 31
Ajan 36, Yaş: 63
Ajan 37, Yaş: 40
Ajan 38, Yaş: 46
Ajan 39, Yaş: 48
Ajan 40, Yaş: 53
Ajan 41, Yaş: 75
Ajan 42, Yaş: 34
Ajan 43, Yaş: 31


In [8]:
# Ajanlara özellik atama

jobs = ["Çiftçi", "Doktor", "Mühendis", "Öğretmen"]

for agent in model.agents:
    agent.job = random.choice(jobs)  # Rastgele meslek ata
    print(f"Ajan {agent.unique_id}, Meslek: {agent.job}")

Ajan 1, Meslek: Doktor
Ajan 2, Meslek: Doktor
Ajan 3, Meslek: Doktor
Ajan 4, Meslek: Doktor
Ajan 5, Meslek: Mühendis
Ajan 6, Meslek: Öğretmen
Ajan 7, Meslek: Çiftçi
Ajan 8, Meslek: Çiftçi
Ajan 9, Meslek: Çiftçi
Ajan 10, Meslek: Çiftçi
Ajan 11, Meslek: Öğretmen
Ajan 12, Meslek: Çiftçi
Ajan 13, Meslek: Mühendis
Ajan 14, Meslek: Doktor
Ajan 15, Meslek: Doktor
Ajan 16, Meslek: Mühendis
Ajan 17, Meslek: Doktor
Ajan 18, Meslek: Çiftçi
Ajan 19, Meslek: Doktor
Ajan 20, Meslek: Mühendis
Ajan 21, Meslek: Çiftçi
Ajan 22, Meslek: Mühendis
Ajan 23, Meslek: Mühendis
Ajan 24, Meslek: Öğretmen
Ajan 25, Meslek: Çiftçi
Ajan 26, Meslek: Öğretmen
Ajan 27, Meslek: Çiftçi
Ajan 28, Meslek: Mühendis
Ajan 29, Meslek: Çiftçi
Ajan 30, Meslek: Çiftçi
Ajan 31, Meslek: Mühendis
Ajan 32, Meslek: Mühendis
Ajan 33, Meslek: Doktor
Ajan 34, Meslek: Doktor
Ajan 35, Meslek: Doktor
Ajan 36, Meslek: Öğretmen
Ajan 37, Meslek: Öğretmen
Ajan 38, Meslek: Doktor
Ajan 39, Meslek: Çiftçi
Ajan 40, Meslek: Mühendis
Ajan 41, Meslek: 

In [9]:
# Belirli Koşullara Göre Özellik Atama

for agent in model.agents:
    if agent.wealth > 10:
        agent.status = "Zengin"
    else:
        agent.status = "Fakir"
    print(f"Ajan {agent.unique_id}, Durum: {agent.status}")

Ajan 1, Durum: Zengin
Ajan 2, Durum: Zengin
Ajan 3, Durum: Zengin
Ajan 4, Durum: Fakir
Ajan 5, Durum: Zengin
Ajan 6, Durum: Fakir
Ajan 7, Durum: Zengin
Ajan 8, Durum: Zengin
Ajan 9, Durum: Fakir
Ajan 10, Durum: Zengin
Ajan 11, Durum: Zengin
Ajan 12, Durum: Zengin
Ajan 13, Durum: Zengin
Ajan 14, Durum: Zengin
Ajan 15, Durum: Zengin
Ajan 16, Durum: Zengin
Ajan 17, Durum: Zengin
Ajan 18, Durum: Zengin
Ajan 19, Durum: Zengin
Ajan 20, Durum: Zengin
Ajan 21, Durum: Fakir
Ajan 22, Durum: Zengin
Ajan 23, Durum: Zengin
Ajan 24, Durum: Zengin
Ajan 25, Durum: Zengin
Ajan 26, Durum: Zengin
Ajan 27, Durum: Zengin
Ajan 28, Durum: Zengin
Ajan 29, Durum: Zengin
Ajan 30, Durum: Fakir
Ajan 31, Durum: Zengin
Ajan 32, Durum: Zengin
Ajan 33, Durum: Zengin
Ajan 34, Durum: Zengin
Ajan 35, Durum: Zengin
Ajan 36, Durum: Zengin
Ajan 37, Durum: Zengin
Ajan 38, Durum: Zengin
Ajan 39, Durum: Fakir
Ajan 40, Durum: Zengin
Ajan 41, Durum: Zengin
Ajan 42, Durum: Zengin
Ajan 43, Durum: Zengin


In [10]:
# Zengin Ajanları Bulma

rich_agents = [agent for agent in model.agents if agent.wealth > 50]
print(f"Zengin ajan sayısı: {len(rich_agents)}")

Zengin ajan sayısı: 17


In [11]:
# Belirli Meslekteki Ajanları Sayma

doctors = [agent for agent in model.agents if agent.job == "Doktor"]
print(f"Doktor sayısı: {len(doctors)}")


Doktor sayısı: 12


In [13]:
# Tüm ajanları liste olarak al
all_agents = list(model.agents)  # Veya direkt self.schedule.agents



In [14]:
# Griddeki ajanlara erişim

# (x,y) koordinatındaki ajanları al
cell_agents = model.grid.get_cell_list_contents([(3, 5)])

# Tüm grid'deki ajanları listele
all_grid_agents = [agent for agent in model.grid.coord_iter()]

In [15]:
# özellik filtreleme

rich_agents = [a for a in model.agents if a.wealth > 10]
print(f"Zengin ajan sayısı: {len(rich_agents)}")

Zengin ajan sayısı: 37


In [18]:
# Rastgele Ajan Seçme

import random
random_agent = random.choice(list(model.agents))
print(f"Rastgele seçilen ajanın ID: {random_agent.unique_id}")

Rastgele seçilen ajanın ID: 38


In [20]:
# Belirli Bir Türdeki Ajanlar (Multi-Agent Sistemlerde)


class Worker(mesa.Agent): pass
class Consumer(mesa.Agent): pass

worker_agents = [a for a in model.agents if isinstance(a, Worker)]
print(f"Çalışan ajan sayısı: {len(worker_agents)}")

Çalışan ajan sayısı: 0


In [29]:
# Belirli bir ajanın özelliğini değiştir
agent = next(a for a in model.agents if a.unique_id == 3)
agent.wealth = 100

# next() fonksiyonu Mesa'ya özgü bir fonksiyon değildir.
# next() fonksiyonu Python'ın yerleşik (built-in) bir fonksiyonudur ve iteratörlerden
# bir sonraki değeri almak için kullanılır.

# Mesa'da next() fonksiyonunu genellikle ajanları veya diğer iteratör nesneleri üzerinde
# dolaşmak için kullanabilirsin

"""
next() fonksiyonu Mesa'ya özgü bir fonksiyon değildir. next() fonksiyonu Python'ın yerleşik (built-in)
bir fonksiyonudur ve iteratörlerden bir sonraki değeri almak için kullanılır.

Mesa'da next() fonksiyonunu genellikle ajanları veya diğer iteratör nesneleri üzerinde dolaşmak için
kullanabilirsin
"""

In [34]:
# Modelinize ajanlara özel erişim sağlayan metotlar ekle ve aşağıdaki gibi çağır
rich_agents = model.get_agents_by_wealth(min_wealth=50)
print(f"Zengin ajan sayısı: {len(rich_agents)}")
print(f"Toplam ajan: {model.count_agents()}")

Zengin ajan sayısı: 23
Toplam ajan: 49


In [35]:
# Liste Yazdırma (Tüm Ajanların ID'leri)
all_agents = list(model.agents)
print("Tüm ajan ID'leri:", [agent.unique_id for agent in all_agents])

Tüm ajan ID'leri: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


In [43]:
# Pandas DataFrame ile Profesyonel Çıktı

import pandas as pd

agent_data = []
for agent in model.agents:
    agent_data.append({
        'ID': agent.unique_id,
        'Servet': getattr(agent, 'wealth', None),
        'Yaş': getattr(agent, 'age', None),
        'Meslek': getattr(agent, 'job', None),
        # Check if agent.pos is not None before accessing its elements
        'X': agent.pos[0] if agent.pos is not None else None,
        'Y': agent.pos[1] if agent.pos is not None else None
    })

df = pd.DataFrame(agent_data)
print(df)

    ID  Servet   Yaş Meslek    X    Y
0    1      20  None   None  4.0  6.0
1    2      15  None   None  4.0  3.0
2    3      46  None   None  3.0  9.0
3    4      20  None   None  7.0  4.0
4    5      39  None   None  6.0  6.0
5    6      40  None   None  0.0  9.0
6    7      64  None   None  7.0  3.0
7    8      99  None   None  9.0  8.0
8    9      34  None   None  3.0  6.0
9   10      78  None   None  7.0  5.0
10  11      65  None   None  NaN  NaN
11  12      65  None   None  NaN  NaN
12  13      12  None   None  NaN  NaN
13  14      74  None   None  NaN  NaN
14  15      70  None   None  NaN  NaN
15  16      45  None   None  NaN  NaN
16  17      53  None   None  NaN  NaN
17  18      84  None   None  NaN  NaN
18  19      93  None   None  NaN  NaN
19  20      90  None   None  NaN  NaN
20  21      75  None   None  NaN  NaN
21  22      72  None   None  NaN  NaN
22  23       3  None   None  NaN  NaN
23  24      74  None   None  NaN  NaN
24  25      54  None   None  NaN  NaN
25  26      

In [44]:
# Belirli Bir Özelliğe Göre Filtreleme

print("Zengin Ajanlar (wealth > 50):")
rich_agents = [a for a in model.agents if getattr(a, 'wealth', 0) > 50]
for agent in rich_agents:
    print(f"ID: {agent.unique_id}, Servet: {agent.wealth}")

Zengin Ajanlar (wealth > 50):
ID: 7, Servet: 64
ID: 8, Servet: 99
ID: 10, Servet: 78
ID: 11, Servet: 65
ID: 12, Servet: 65
ID: 14, Servet: 74
ID: 15, Servet: 70
ID: 17, Servet: 53
ID: 18, Servet: 84
ID: 19, Servet: 93
ID: 20, Servet: 90
ID: 21, Servet: 75
ID: 22, Servet: 72
ID: 24, Servet: 74
ID: 25, Servet: 54
ID: 27, Servet: 85
ID: 32, Servet: 77
ID: 35, Servet: 84
ID: 42, Servet: 74
ID: 47, Servet: 72
ID: 48, Servet: 67
ID: 49, Servet: 51
